In [ ]:
# Configuración COSMIC v0.0.1 - Estructura Modular
import sys
from pathlib import Path

# Configuración automática de rutas relativas
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parents[2]  # Tres niveles arriba desde data/test/NGC6383/

# Agregar al path si no está
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"COSMIC v0.0.1 - NGC6383 Analysis")
print(f"Directorio actual: {CURRENT_DIR}")
print(f"Proyecto: {PROJECT_ROOT}")

# Verificar instalación de COSMIC
try:
    import cosmic
    print("COSMIC modular disponible")
except ImportError:
    print("Instalando COSMIC...")
    import os
    os.system(f"pip install -e {PROJECT_ROOT}")
    import cosmic
    print("COSMIC instalado")

In [ ]:
# Imports usando la nueva estructura modular
from cosmic.analysis.analyzer import ClusterAnalyzer

# El import legacy sigue funcionando para compatibilidad:
# from COSMIC import ClusterAnalyzer  # Funciona igual

print(f"ClusterAnalyzer importado: {ClusterAnalyzer}")
print(f"Listo para análisis con nueva API modular")

In [ ]:
# Configuración de datos usando rutas relativas
data_folder = 'data/40/'
file_path = data_folder + 'clustering_results.dill'

print(f"Carpeta de datos: {data_folder}")
print(f"Archivo: {file_path}")
print(f"Existe: {Path(file_path).exists()}")

# Inicializar el analizador con la nueva API
ca = ClusterAnalyzer(file_path)
print("ClusterAnalyzer inicializado correctamente")

In [ ]:
# 3) Build (or retrieve) a summary table: one row per cluster, 
#    with columns for label, n_members, persistence, etc.
ca.clusters_summary(include_noise=True)

In [ ]:
ca.plot_persistence_vs_members(percentile=0.8, figsize=(10,10))

In [ ]:
cluster_id = 12
cluster_data = ca.select_cluster(cluster_id)  # esto devuelve solo los objetos de ese cluster
print(f"Cluster {cluster_id} selected, contains {len(cluster_data)} sources.")

In [ ]:
ca.plot_probability_vs_gmag(figsize=(8,8)) # a esto no se la ha hecho nada, es el cluster tal cual como slió del preprocessing.

In [ ]:
pmin = 0.5
pre = (ca.data['probability_hdbscan'] >= pmin)

In [ ]:
ca.sigma_clip_parallax(
    sigma=2.0,
    use_biweight=True,
    preselector_mask=pre,
    print_results=True,
    in_place=True
)

In [ ]:
ca.plot_probability_vs_gmag(figsize=(8,8))

In [ ]:
ca.pms_characterization(
    cluster=cluster_id,   # usa el mismo cluster_id que seleccionaste (ej: 32)
    run_cli=True
)

In [ ]:
ca.plot_pms(
    cluster=cluster_id,
    pms_threshold=0.6,    # ajusta si deseas un corte distinto
    figsize=(7,7),
    layout="tight",
)

In [ ]:
cluster_data = ca.select_cluster(cluster_id)

In [ ]:
# ── Isochrone fitting ────────────────────────────────────────────────────────
fitter, idata = ca.fit_isochrone(
    cluster=cluster_id,
    isochs_path="./MIST/",
    dm_mu=10.2,
    dm_sigma=0.3,
    dm_range=(9.5, 10.7),
    loga_range=(6.0, 7.0),
    Av_range=(0.0, 3.0),
    M_met=200,
    M_loga=200,
    grid_cache="./data/40/hgrid.npz",
    draws=2000,
    tune=1000,
    chains=4,
    target_accept=0.95,
    nuts_sampler="blackjax",
)

In [ ]:
import arviz as az

# Resumen numérico de las 4 params principales
az.summary(idata, var_names=["met","loga","dm","Av"], kind="stats", round_to=4)

In [ ]:
# Posterior univariado
az.plot_posterior(idata, var_names=["met","loga","dm","Av"], hdi_prob=0.94);

In [ ]:
# Corner plot (pares), con KDE
az.plot_pair(
    idata,
    var_names=["met","loga","dm","Av"],
    kind="kde",  # o "scatter"
    divergences="bottom",
    marginals=True
);

In [ ]:
# Trazas por cadena (útil para ver mezcla)
az.plot_trace(idata, var_names=["met","loga","dm","Av"]);

In [ ]:
import numpy as np

post = idata.posterior

met_m  = np.median(np.asarray(post["met"]).reshape(-1))
loga_m = np.median(np.asarray(post["loga"]).reshape(-1))
dm_m   = np.median(np.asarray(post["dm"]).reshape(-1))
Av_m   = np.median(np.asarray(post["Av"]).reshape(-1))

best = {"met": float(met_m),
        "loga": float(loga_m),
        "dm": float(dm_m),
        "Av": float(Av_m)}

print(best)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Single posterior draw → synthetic CMD
obs_mag, obs_col, cmds = fitter.posterior_cmd(idata, num_samples=1)
syn = cmds[0]  # [G, BP-RP, ...]

plt.figure(figsize=(5, 6))
plt.scatter(obs_col, obs_mag, s=8, alpha=0.35, label="Obs")
plt.scatter(syn[1], syn[0], s=8, alpha=0.35, label="Synth @ posterior draw")
plt.gca().invert_yaxis()
plt.xlabel("BP−RP")
plt.ylabel("G")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import arviz as az

az.summary(idata, hdi_prob=0.95)

In [ ]:
post_median = {v: np.median(idata.posterior[v].values.flatten())
               for v in ["met", "dm", "loga", "Av"]}
print(post_median)

In [ ]:
import matplotlib.pyplot as plt

obs_mag, obs_col, cmds = fitter.posterior_cmd(idata, num_samples=20)

plt.figure()
for i, syn in enumerate(cmds):
    plt.scatter(syn[1], syn[0], s=3, alpha=0.3, label="posterior draw" if i == 0 else "")

plt.gca().invert_yaxis()
plt.legend()
plt.show()

In [ ]:
print(az.summary(idata, var_names=["met","loga","dm","Av"], kind="stats"))

# --- percentiles útiles ---
p = az.extract(idata, var_names=["met","loga","dm","Av"]).to_dataframe()
p50 = p.median()  # mediana marginal (baseline)
best_now = dict(met=float(p50["met"]), loga=float(p50["loga"]),
                dm=float(p50["dm"]),  Av=float(p50["Av"]))
print("Baseline (mediana):", best_now)

In [ ]:
import matplotlib.pyplot as plt

obs_mag, obs_col, cmds = fitter.posterior_cmd(idata, num_samples=1)
syn = cmds[0]
x_syn, y_syn = syn[1], syn[0]

plt.figure(figsize=(6, 7))
plt.scatter(obs_col, obs_mag, s=7, alpha=0.5, label="Obs", zorder=2)
plt.scatter(x_syn, y_syn, s=7, alpha=0.5, label="Synth @draw", zorder=3)
plt.gca().invert_yaxis()
plt.xlabel("BP−RP"); plt.ylabel("G"); plt.legend(); plt.title("Baseline CMD")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fast_histogram import histogram2d
import pytensor.tensor as pt

Nb_mag, Nb_col = fitter._Nbins
mag_min, mag_max = fitter._mag_range
col_min, col_max = fitter._col_range

# Observed Hess diagram
cl_histo_full = histogram2d(
    fitter._obs_mag, fitter._obs_col,
    bins=[Nb_mag, Nb_col],
    range=[[mag_min, mag_max], [col_min, col_max]],
).astype("float64")

# Synthetic Hess from posterior median
post_vals = {v: float(np.median(idata.posterior[v].values.flatten()))
             for v in ["met", "loga", "dm", "Av"]}
H0 = fitter._interp_H(post_vals["met"], post_vals["loga"])
dmag = post_vals["dm"] + fitter._kG * post_vals["Av"]
dcol = fitter._k_col1 * post_vals["Av"]
Hsyn = fitter._shift_histogram(H0, dmag, dcol).eval()

fig, ax = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)
ax[0].imshow(cl_histo_full.T, origin="lower", aspect="auto")
ax[0].set_title("Obs bins")
ax[1].imshow(Hsyn.T, origin="lower", aspect="auto")
ax[1].set_title("Synth bins @mediana")
diff = cl_histo_full - Hsyn
ax[2].imshow(diff.T, origin="lower", aspect="auto")
ax[2].set_title("Obs − Synth")
for a in ax:
    a.set_xlabel("mag bin"); a.set_ylabel("color bin")
plt.tight_layout(); plt.show()

In [ ]:
print(syn[0].min(), syn[0].max())   # magnitudes sintéticas
print(syn[1].min(), syn[1].max())   # colores sintéticos

In [ ]:
print(fitter._obs_mag.min(), fitter._obs_mag.max())
print(fitter._obs_col.min(), fitter._obs_col.max())

In [ ]:
# loglike promedio (del sample)
llk = idata.sample_stats["lp"].mean().item()
print(f"logp medio (baseline): {llk:.1f}")

# LOO/WAIC si usaste Poisson observado:
try:
    loo_base = az.loo(idata)
    waic_base = az.waic(idata)
    print("LOO (baseline):", loo_base)
    print("WAIC(baseline):", waic_base)
except Exception as e:
    print("LOO/WAIC no disponible para esta estructura:", e)